# Holdout Evaluation

Evaluate the selected edit on holdout prompts and inspect topic-level failure patterns and sample responses.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("FRS_REPO_URL", "").strip()
REPO_DIR = Path("/kaggle/working/false-refusal-suppression")

if not REPO_URL:
    raise RuntimeError("Set FRS_REPO_URL in the Kaggle environment before running this notebook.")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(REPO_DIR)

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-4B"
SPLIT_DIR = Path("data/processed/splits/sample_small")
DIRECTION_ARTIFACT = Path("artifacts/directions/qwen3_sample_borderline_vs_easy.json")
SEARCH_ARTIFACT = Path("artifacts/edits/qwen3_sample_search.json")
EVAL_ARTIFACT = Path("artifacts/eval/qwen3_sample_holdout.json")

print(MODEL_ID)
print(EVAL_ARTIFACT)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/run_eval.py",
        "--model-id",
        MODEL_ID,
        "--prompts",
        str(SPLIT_DIR / "holdout.jsonl"),
        "--direction-artifact",
        str(DIRECTION_ARTIFACT),
        "--candidate-json",
        str(SEARCH_ARTIFACT),
        "--output",
        str(EVAL_ARTIFACT),
    ],
    check=True,
 )

In [ ]:
import json
import pandas as pd

with open(EVAL_ARTIFACT, "r", encoding="utf-8") as handle:
    report = json.load(handle)

display(pd.DataFrame([report["metrics"]]))
display(pd.DataFrame.from_dict(report["topic_breakdown"], orient="index").sort_values("refusal_rate", ascending=False))
display(pd.DataFrame(report["response_samples"]))